In [1]:
# 5_add_nl_strings.ipynb
#
# Reads data/4_feature_eng/{WAVE}_feature_eng.pkl (4a; optional KEEP/5a may add columns),
# builds nl_profile per row, writes data/5_add_nl_strings/{WAVE}_with_nl_profile.pkl + .csv,
# and updates the 4_feature_eng pickle (step 6 reads nl_profile from there).

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_variables as _cv
_cv.reload_config_variables()
from data_pipeline.config_variables import DATA_FOLDER, VARIABLES, WAVE

import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
FEATURE_ENG_PKL = Path(f"../{DATA_FOLDER}/4_feature_eng/{WAVE}_feature_eng.pkl")
OUTPUT_DIR      = Path(f"../{DATA_FOLDER}/5_add_nl_strings")
OUTPUT_PKL      = OUTPUT_DIR / f"{WAVE}_with_nl_profile.pkl"
OUTPUT_CSV      = OUTPUT_DIR / f"{WAVE}_with_nl_profile.csv"

# ── Load ──────────────────────────────────────────────────────────────────────
print(f"Reading: {FEATURE_ENG_PKL}")
df = pd.read_pickle(FEATURE_ENG_PKL)
print(f"Loaded {len(df):,} individuals × {len(df.columns)} columns")

# ── Decoders ──────────────────────────────────────────────────────────────────
_CURRENT_YEAR = pd.Timestamp.now().year
_SKIP_LABELS  = {"not provided", "not applicable", "inapplicable", "missing",
                 "refusal", "don't know", "proxy", "not relevant", "no answer"}


def decode_age_eng(val) -> str | None:
    """Age in years from step 4a ``{WAVE}_doby_dv_eng`` (birth_year_to_age)."""
    if pd.isna(val):
        return None
    fval = float(val)
    if fval < 0:
        return None
    age = int(round(fval))
    return str(age) if 0 < age < 120 else None


def decode(base_code: str, val) -> str | None:
    if pd.isna(val):
        return None
    fval = float(val)
    if fval < 0:
        return None
    if base_code == "doby_dv":
        age = _CURRENT_YEAR - int(fval)
        return str(age) if 0 < age < 120 else None
    cat_map = VARIABLES.get(base_code, {}).get("categories")
    if cat_map:
        label = cat_map.get(fval)
        if label and label.lower().strip() not in _SKIP_LABELS:
            return label
        return None
    return str(int(round(fval)))


def build_profile(row: pd.Series) -> str:
    age = decode_age_eng(row.get(f"{WAVE}_doby_dv_eng"))
    if age is None:
        age = decode("doby_dv", row.get(f"{WAVE}_doby_dv"))
    ethnicity= decode("racel_dv",  row.get(f"{WAVE}_racel_dv"))
    sex      = decode("sex_dv",    row.get(f"{WAVE}_sex_dv"))
    hiqual   = decode("hiqual_dv", row.get(f"{WAVE}_hiqual_dv"))
    jbstat   = decode("jbstat",    row.get(f"{WAVE}_jbstat"))
    marstat  = decode("marstat_dv",row.get(f"{WAVE}_marstat_dv"))
    tenure   = decode("tenure_dv", row.get(f"{WAVE}_tenure_dv"))
    hhtype   = decode("hhtype_dv", row.get(f"{WAVE}_hhtype_dv"))
    health   = decode("scsf1",     row.get(f"{WAVE}_scsf1"))

    age_str = f"{age} year old" if age else "unknown age"
    eth_str = f"{ethnicity} " if ethnicity else ""
    sex_str = sex.lower() if sex else "person"
    s1 = f"A {age_str} {eth_str}{sex_str}"

    s2 = f"whose highest qualification is {hiqual.lower()}" if hiqual else None
    s3 = f"Their employment status is {jbstat.lower()}" if jbstat else None
    s4 = f"their marital status is {marstat.lower()}" if marstat else None
    s5 = f"their housing tenure is {tenure.lower()}" if tenure else None
    s6 = f"and they live in a {hhtype.lower()} household" if hhtype else None
    s7 = f"They rate their general health as {health.lower()}" if health else None

    middle = ", ".join(p for p in [s2, s3, s4, s5, s6] if p)
    profile = s1 + (f", {middle}." if middle else ".")
    if s7:
        profile += f" {s7}."
    return profile


# Sanity check
print("\nSample profile:")
print(build_profile(df.iloc[0]))

# ── Generate & save ─────────────────────────────────────────────────────────
tqdm.pandas(desc="Building profiles")
df["nl_profile"] = df.progress_apply(build_profile, axis=1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_pickle(OUTPUT_PKL, protocol=5)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(df):,} rows to {OUTPUT_PKL}")
print(f"CSV saved to {OUTPUT_CSV}")

FEATURE_ENG_PKL.parent.mkdir(parents=True, exist_ok=True)
df.to_pickle(FEATURE_ENG_PKL, protocol=5)
print(f"Updated feature-eng pickle (for step 6): {FEATURE_ENG_PKL}")
display(df[["pidp", "nl_profile"]].head(5))

Reading: ../data/4_feature_eng/k_feature_eng.pkl
Loaded 27,330 individuals × 41 columns

Sample profile:
A 57 year old White: British/English/Scottish/Welsh/N. Irish female, whose highest qualification is gcse etc, Their employment status is unemployed, their marital status is married/civil partner. They rate their general health as fair.


Building profiles: 100%|██████████| 27330/27330 [00:00<00:00, 58892.00it/s]



Saved 27,330 rows to ../data/5_add_nl_strings/k_with_nl_profile.pkl
CSV saved to ../data/5_add_nl_strings/k_with_nl_profile.csv
Updated feature-eng pickle (for step 6): ../data/4_feature_eng/k_feature_eng.pkl


,pidp,nl_profile
0,68006127,A 57 year old White: British/English/Scottish/...
1,68020564,A 56 year old White: British/English/Scottish/...
2,68008847,A 69 year old White: British/English/Scottish/...
3,68009527,A 49 year old White: British/English/Scottish/...
4,68061288,A 40 year old White: British/English/Scottish/...
